Preparación de datos y primer modelo

En este notebook se prepararán los datos según lo diagnosticado en la etapa de análisis (limpieza de valores vacíos, transformación a números y separación de datos) para entrenar un primer modelo de prueba.

In [19]:
# Para importar las herramientas
import pandas as pd
from sklearn.model_selection import train_test_split

In [20]:
# Para cargar el archivo original
df = pd.read_csv("../data/customer_churn_historical.csv")

# Para ver la cantidad de filas y columnas
df.shape

(7043, 21)

In [21]:
# Para quitar columnas que no me sirven
# Para separar los datos de los clientes de la respuesta que queremos predecir (1 se fue, 0 se quedó)
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"Yes": 1, "No": 0})

In [22]:
# Para separar en 80% para entrenar y 20% para probar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [23]:
# Para ver cuántos clientes quedaron para cada parte
print(f"Para entrenar: {X_train.shape[0]}")
print(f"Para probar: {X_test.shape[0]}")

Para entrenar: 5634
Para probar: 1409


* Se descartó "customerID" porque es un código que no aporta información para predecir el comportamiento del cliente.
* Se descartó "Churn" en X para que el modelo no tenga la respuesta de antemano, y aprenda a deducirla a partir del resto de los datos.
* Se separó "Churn" como la columna objetivo a predecir, transformando sus valores a 1 (se fue) y 0 (se quedó).
* Se dividieron los datos en 5634 filas para entrenamiento (80%) y 1409 para prueba (20%), manteniendo en ambas partes la misma proporción de clientes que cancelaron el servicio.

In [24]:
# Para convertir TotalCharges a número y rellenar los vacíos de clientes nuevos con 0
X_train["TotalCharges"] = pd.to_numeric(X_train["TotalCharges"], errors="coerce").fillna(0)
X_test["TotalCharges"] = pd.to_numeric(X_test["TotalCharges"], errors="coerce").fillna(0)

In [25]:
# Para comprobar que no quedaron valores vacíos en ninguna de las dos partes
print("Vacíos en entrenamiento:", X_train["TotalCharges"].isnull().sum())
print("Vacíos en prueba:       ", X_test["TotalCharges"].isnull().sum())

Vacíos en entrenamiento: 0
Vacíos en prueba:        0


In [26]:
# Para transformar las columnas de texto en columnas numéricas de 1 y 0
X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, drop_first=True, dtype=int)

In [27]:
# Para forzar a que el conjunto de prueba tenga exactamente los mismos nombres y orden de columnas que el de entrenamiento
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [28]:
# Para consultar cuántas columnas quedaron en cada conjunto
print("Columnas en entrenamiento:", X_train.shape[1])
print("Columnas en prueba:       ", X_test.shape[1])

Columnas en entrenamiento: 30
Columnas en prueba:        30


In [31]:
# Para importar el escalador y el modelo
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [32]:
# Para ajustar la escala de las columnas usando únicamente los datos de entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [33]:
# Para crear y entrenar el modelo base sobre los datos escalados
modelo_base = LogisticRegression(random_state=42)
modelo_base.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [34]:
# Para generar las predicciones sobre el conjunto de prueba
y_pred = modelo_base.predict(X_test_scaled)

In [35]:
# Para importar las herramientas de evaluación
from sklearn.metrics import classification_report, confusion_matrix

In [36]:
# Para mostrar la matriz de confusión (filas: valor real, columnas: predicción)
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

Matriz de confusión:
[[951  86]
 [205 167]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1037
           1       0.66      0.45      0.53       372

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409



Interpretación del modelo base (Regresión Logística)

* Exactitud general (79%): Es un número engañoso porque la gran mayoría de los clientes no se va; si dijéramos que nadie se va nunca, ya acertaríamos el 74% de las veces sin hacer ningún cálculo.
* Detección de cancelaciones (45%): De los 372 clientes que efectivamente se fueron, el modelo solo identificó a 167 y pasó por alto a 205. Para el objetivo de retener clientes, este rendimiento es insuficiente.
* Precisión en alertas (66%): Cuando el modelo señala que alguien se va a ir, acierta en 2 de cada 3 casos (86 falsas alarmas).


* Conclusión: Este modelo sirve como punto de partida básico. El objetivo principal en los siguientes intentos es lograr que se le escapen muchos menos clientes que cancelan, sin llenar a la empresa de falsas alarmas.